In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, mean_absolute_error, mean_squared_error, r2_score, roc_auc_score
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
from torch.utils.data import Dataset
from torch.utils.data import DataLoader

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

from tqdm import tqdm
import json
import time
import os
import random

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
df = pd.read_parquet("Data/dipser_dataset.parquet", engine="pyarrow")
df["age_group"] = pd.cut(df["age"], bins=[13, 20, 22, 26, 44]) # bins=[13, 17, 20, 22, 26, 44]
df.head()

In [ ]:
df.shape

In [ ]:
SEED = 42
SEQUENCE_LENGTH = 10
MAX_MISSING_VISUAL_FRAMES = SEQUENCE_LENGTH - 1

# Label-imbalance intervention toggles.
USE_WEIGHTED_LOSS = True
USE_WEIGHTED_SAMPLER = False
WEIGHT_ALPHA = 0.5

# Split stratification is kept fixed for paired comparisons. Prefer "gender_age_group"
# if every joint cell has enough subjects; otherwise use the thesis baseline split and
# treat fairness conclusions as paired within that split only.
STRATIFY_COLUMN = "gender"
BATCH_SIZE = 32
NUM_WORKERS = 8

ATTENTION_BINS = [2, 2.5, 3, 3.5, 4.75]


In [ ]:
df[['attention', 'image_path', 'age', 'gender']].isna().sum()

In [ ]:
sensor_cols = [
    col for col in df.columns
    if (
       # "sensor" in col.lower()
        "acceleration" in col.lower()
        or "gyro" in col.lower()
      #  or "rotation" in col.lower()
        or "heart" in col.lower()
      #  or "light" in col.lower()
        or "accel" in col.lower()
    )
    and "std" not in col.lower()
]

# Heart rate is physiologically different from motion/device sensors, so Fusion
# follows Temporal Sensor and gives it a separate projection stream.
hr_cols = ["heart_rate"] if "heart_rate" in sensor_cols else []
motion_cols = [col for col in sensor_cols if col not in hr_cols]
hr_indices = [sensor_cols.index(col) for col in hr_cols]
motion_indices = [sensor_cols.index(col) for col in motion_cols]

len(sensor_cols), len(motion_cols), len(hr_cols), sensor_cols[:5]

In [ ]:
# Treat physiologically impossible HR values as missing before missing flags and scaling.
# The smartwatch can emit 0, which should not be interpreted as a real heart rate.
df.loc[df["heart_rate"] < 30, "heart_rate"] = np.nan

df[sensor_cols] = df[sensor_cols].astype(np.float32)

In [ ]:
# keep only 1 frame per second (the first)
df_sec = (df.sort_values(["subject_experiment_id", "time_sec"])
      .groupby(["subject_experiment_id", "time_sec"])
      .first()
      .reset_index())
df_sec.shape

In [ ]:
reconstructed_sequences = []

for sequence_id, sequence_df in df_sec.groupby("subject_experiment_id"):
    sequence_df = sequence_df.sort_values("time_sec")

    complete_seconds = pd.DataFrame({
        "time_sec": np.arange(
            sequence_df["time_sec"].min(),
            sequence_df["time_sec"].max() + 1
        )
    })

    reconstructed = complete_seconds.merge(
        sequence_df,
        on="time_sec",
        how="left"
    )

    reconstructed["subject_experiment_id"] = sequence_id
    reconstructed_sequences.append(reconstructed)

temporal_frame_dataset = pd.concat(reconstructed_sequences, ignore_index=True)

sequence_metadata = (
    df[["subject_experiment_id", "subject_id", "gender", "age", "age_group"]]
    .drop_duplicates("subject_experiment_id"))

# Restore metadata for reconstructed missing seconds. Sensor/image/target values
# stay missing unless observed, so missingness flags remain meaningful.
temporal_frame_dataset = temporal_frame_dataset.drop(
    columns=["subject_id", "gender", "age", "age_group"],
    errors="ignore").merge(sequence_metadata, on="subject_experiment_id", how="left")

temporal_frame_dataset["visual_missing"] = temporal_frame_dataset["image_path"].isna().astype(np.float32)
temporal_frame_dataset["motion_missing"] = temporal_frame_dataset[motion_cols].isna().all(axis=1).astype(np.float32)
temporal_frame_dataset["hr_missing"] = temporal_frame_dataset[hr_cols].isna().all(axis=1).astype(np.float32) if hr_cols else 1.0
temporal_frame_dataset["sensor_missing"] = temporal_frame_dataset[sensor_cols].isna().all(axis=1).astype(np.float32)
temporal_frame_dataset["sensor_partial_nan"] = (
    temporal_frame_dataset[sensor_cols].isna().any(axis=1)
    & ~temporal_frame_dataset[sensor_cols].isna().all(axis=1)
).astype(np.float32)

temporal_frame_dataset[["subject_experiment_id", "time_sec", "visual_missing", "motion_missing", "hr_missing", "sensor_missing", "attention"]].head()

In [ ]:
temporal_frame_dataset.shape

In [ ]:
# feature_index = pd.read_parquet("Data/resnet50_features/resnet50_frame_index.parquet")
# feature_store_path = "Data/resnet50_features/resnet50_frame_features.npy"
# VISUAL_FEATURE_DIM = 2048

# feature_index = pd.read_parquet('Data/All_clip_vitl14_features/All_clip_vitl14_frame_index.parquet')
# feature_store_path =  'Data/All_clip_vitl14_features/All_clip_vitl14_frame_features.npy'
# VISUAL_FEATURE_DIM = 768

feature_index = pd.read_parquet('Data/clip_vitl14_features/clip_vitl14_frame_index.parquet')
feature_store_path =  'Data/clip_vitl14_features/clip_vitl14_frame_features.npy'
VISUAL_FEATURE_DIM = 768

feature_row_by_path = dict(zip(feature_index["image_path"], feature_index["feature_row"]))
FEATURES = feature_store_path.split('/')[-2]

temporal_frame_dataset["feature_row"] = (
    temporal_frame_dataset["image_path"]
    .map(feature_row_by_path)
    .fillna(-1)
    .astype(np.int64))

missing_feature_rows = (
    (temporal_frame_dataset["visual_missing"] == 0)
    & (temporal_frame_dataset["feature_row"] == -1)
).sum()
print("Non-missing image rows without cached features:", missing_feature_rows)

In [ ]:
subject_df = temporal_frame_dataset[["subject_id", STRATIFY_COLUMN]].dropna(subset=[STRATIFY_COLUMN]).drop_duplicates()

train_sub, temp_sub = train_test_split(
    subject_df,
    test_size=0.3,
    stratify=subject_df[STRATIFY_COLUMN],
    random_state=SEED
)

val_sub, test_sub = train_test_split(
    temp_sub,
    test_size=0.5,
    stratify=temp_sub[STRATIFY_COLUMN],
    random_state=SEED
)

train_subjects = train_sub["subject_id"]
val_subjects = val_sub["subject_id"]
test_subjects = test_sub["subject_id"]

train_frames = temporal_frame_dataset[temporal_frame_dataset["subject_id"].isin(train_subjects)].copy()
val_frames = temporal_frame_dataset[temporal_frame_dataset["subject_id"].isin(val_subjects)].copy()
test_frames = temporal_frame_dataset[temporal_frame_dataset["subject_id"].isin(test_subjects)].copy()

# Train-only sensor normalization. Missing values become 0 after standardization;
# motion/hr missing flags tell the model which modality was absent at that timestep.
sensor_means = train_frames.loc[train_frames["sensor_missing"] == 0, sensor_cols].mean()
sensor_stds = train_frames.loc[train_frames["sensor_missing"] == 0, sensor_cols].std()
sensor_stds = sensor_stds.replace(0, np.nan).fillna(1.0)
sensor_means = sensor_means.fillna(0.0)


def apply_sensor_scaling(frame_df):
    frame_df = frame_df.copy()
    scaled = ((frame_df[sensor_cols] - sensor_means) / sensor_stds).astype(np.float32)
    scaled = scaled.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    scaled.loc[frame_df["sensor_missing"] == 1, :] = 0.0

    # Keep each sensor as its own column instead of packing all sensors into
    # one object-valued sensor_vector column. This keeps the sequence dataframe
    # inspectable and aligned with the original df column structure.
    frame_df.loc[:, sensor_cols] = scaled.to_numpy(dtype=np.float32)
    return frame_df


train_frames = apply_sensor_scaling(train_frames)
val_frames = apply_sensor_scaling(val_frames)
test_frames = apply_sensor_scaling(test_frames)

MOTION_INPUT_DIM = len(motion_cols) + 1  # +1 for motion_missing flag
HR_INPUT_DIM = len(hr_cols) + 1          # +1 for hr_missing flag
SENSOR_INPUT_DIM = len(sensor_cols) + 1

pd.DataFrame({
    "split": ["train", "val", "test"],
    "rows": [len(train_frames), len(val_frames), len(test_frames)],
    "subjects": [train_frames.subject_id.nunique(), val_frames.subject_id.nunique(), test_frames.subject_id.nunique()],
    "target_mean": [train_frames.attention.mean(), val_frames.attention.mean(), test_frames.attention.mean()],
    "visual_missing_rate": [train_frames.visual_missing.mean(), val_frames.visual_missing.mean(), test_frames.visual_missing.mean()],
    "motion_missing_rate": [train_frames.motion_missing.mean(), val_frames.motion_missing.mean(), test_frames.motion_missing.mean()],
    "hr_missing_rate": [train_frames.hr_missing.mean(), val_frames.hr_missing.mean(), test_frames.hr_missing.mean()],
    "sensor_missing_rate": [train_frames.sensor_missing.mean(), val_frames.sensor_missing.mean(), test_frames.sensor_missing.mean()],
    "sensor_partial_nan_rate": [train_frames.sensor_partial_nan.mean(), val_frames.sensor_partial_nan.mean(), test_frames.sensor_partial_nan.mean()],
})

In [ ]:
def create_temporal_sequences(frame_df, sequence_length=SEQUENCE_LENGTH):
    sequences = []

    for sequence_id, sequence_df in frame_df.groupby("subject_experiment_id"):
        sequence_df = sequence_df.sort_values("time_sec").reset_index(drop=True)

        if len(sequence_df) < sequence_length:
            continue

        for i in range(sequence_length - 1, len(sequence_df)):
            target = sequence_df.iloc[i]["attention"]

            if pd.isna(target):
                continue

            history = sequence_df.iloc[i - sequence_length + 1:i + 1]
            visual_missing_flags = history["visual_missing"].astype(np.float32).values

            if visual_missing_flags.sum() > MAX_MISSING_VISUAL_FRAMES:
                continue

            motion_missing_flags = history["motion_missing"].astype(np.float32).values
            hr_missing_flags = history["hr_missing"].astype(np.float32).values
            sensor_missing_flags = history["sensor_missing"].astype(np.float32).values

            sequence_record = {
                "subject_experiment_id": sequence_id,
                "subject_id": sequence_df.iloc[i]["subject_id"],
                "gender": sequence_df.iloc[i]["gender"],
                "age": sequence_df.iloc[i]["age"],
                "age_group": sequence_df.iloc[i]["age_group"],
                "time_sec": int(sequence_df.iloc[i]["time_sec"]),
                "feature_rows": history["feature_row"].tolist(),
                "visual_missing_flags": visual_missing_flags.tolist(),
                "motion_missing_flags": motion_missing_flags.tolist(),
                "hr_missing_flags": hr_missing_flags.tolist(),
                "sensor_missing_flags": sensor_missing_flags.tolist(),
                "target": float(target),
            }

            # Store every scaled sensor as its own temporal column. Each value is
            # a length-SEQUENCE_LENGTH list, e.g. train_df["heart_rate"].iloc[0].
            for sensor_col in sensor_cols:
                sequence_record[sensor_col] = (
                    history[sensor_col]
                    .astype(np.float32)
                    .to_numpy(dtype=np.float32)
                    .tolist()
                )

            sequences.append(sequence_record)

    return pd.DataFrame(sequences)

In [ ]:
train_df = create_temporal_sequences(train_frames)
val_df = create_temporal_sequences(val_frames)
test_df = create_temporal_sequences(test_frames)

pd.DataFrame({
    "split": ["train", "val", "test"],
    "sequences": [len(train_df), len(val_df), len(test_df)],
    "subjects": [train_df.subject_id.nunique(), val_df.subject_id.nunique(), test_df.subject_id.nunique()],
    "target_mean": [train_df.target.mean(), val_df.target.mean(), test_df.target.mean()],
})

### Train Test Split

In [ ]:
class MultimodalFusionDataset(Dataset):

    def __init__(self, sequence_df, feature_store_path):
        self.df = sequence_df.reset_index(drop=True)
        self.feature_store_path = feature_store_path
        self.feature_store = None

    def __len__(self):
        return len(self.df)

    def _features(self):
        if self.feature_store is None:
            self.feature_store = np.load(self.feature_store_path, mmap_mode="r")
        return self.feature_store

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        feature_rows = np.asarray(row["feature_rows"], dtype=np.int64)
        visual_features = np.zeros((len(feature_rows), VISUAL_FEATURE_DIM), dtype=np.float32)

        valid = feature_rows >= 0
        visual_features[valid] = self._features()[feature_rows[valid]]

        # Reconstruct the T x sensor_dim tensor from separate temporal sensor
        # columns instead of reading one packed sensor_values column.
        sensors = np.stack(
            [
                np.asarray(row[sensor_col], dtype=np.float32)
                for sensor_col in sensor_cols
            ],
            axis=1
        )
        sensors = torch.tensor(sensors, dtype=torch.float32)
        sensors = torch.nan_to_num(sensors, nan=0.0, posinf=0.0, neginf=0.0)

        motion = sensors[:, motion_indices]
        motion_missing = torch.tensor(row["motion_missing_flags"], dtype=torch.float32).unsqueeze(-1)
        motion = torch.cat([motion, motion_missing], dim=-1)

        if hr_indices:
            heart_rate = sensors[:, hr_indices]
        else:
            heart_rate = torch.zeros((sensors.shape[0], 0), dtype=torch.float32)
        hr_missing = torch.tensor(row["hr_missing_flags"], dtype=torch.float32).unsqueeze(-1)
        heart_rate = torch.cat([heart_rate, hr_missing], dim=-1)

        visual_features = torch.tensor(visual_features, dtype=torch.float32)
        visual_missing_flags = torch.tensor(row["visual_missing_flags"], dtype=torch.float32)
        target = torch.tensor(row["target"], dtype=torch.float32)
        sample_weight = torch.tensor(row["sample_weight"], dtype=torch.float32)

        return visual_features, motion, heart_rate, visual_missing_flags, target, sample_weight, idx

In [ ]:
# Quantify target imbalance and create inverse-frequency train weights.
# Weights are learned from train only; val/test weights are diagnostic only.
def add_attention_bin_weights(alpha, train_df, val_df, test_df):
    train_df = train_df.copy()
    val_df = val_df.copy()
    test_df = test_df.copy()

    train_bins = pd.cut(train_df["target"], bins=ATTENTION_BINS, include_lowest=True)
    bin_counts = train_bins.value_counts().sort_index()
    nonzero_counts = bin_counts[bin_counts > 0]

    bin_weights = (len(train_df) / (len(nonzero_counts) * nonzero_counts)) ** alpha

    for split_df in [train_df, val_df, test_df]:
        split_bins = pd.cut(split_df["target"], bins=ATTENTION_BINS, include_lowest=True)
        split_df["attention_bin"] = split_bins.astype(str)
        split_df["sample_weight"] = split_bins.map(bin_weights).astype(float).fillna(1.0)

    train_weight_mean = train_df["sample_weight"].mean()

    for split_df in [train_df, val_df, test_df]:
        split_df["sample_weight"] = split_df["sample_weight"] / train_weight_mean

    normalized_bin_weights = bin_weights / train_weight_mean

    imbalance_table = pd.DataFrame({
        "bin": bin_counts.index.astype(str),
        "train_count": bin_counts.values,
        "weight": [
            float(normalized_bin_weights.get(idx, np.nan))
            for idx in bin_counts.index
        ],
    })

    return train_df, val_df, test_df, imbalance_table

train_df, val_df, test_df, imbalance_table = add_attention_bin_weights(WEIGHT_ALPHA, train_df, val_df, test_df)
display(imbalance_table)

pd.DataFrame({
    "split": ["train", "val", "test"],
    "sequences": [len(train_df), len(val_df), len(test_df)],
    "subjects": [train_df.subject_id.nunique(), val_df.subject_id.nunique(), test_df.subject_id.nunique()],
    "target_mean": [train_df.target.mean(), val_df.target.mean(), test_df.target.mean()],
    "mean_sample_weight": [train_df.sample_weight.mean(), val_df.sample_weight.mean(), test_df.sample_weight.mean()],
})

In [ ]:
train_df.shape, val_df.shape, test_df.shape

In [ ]:
len(set(train_df['subject_experiment_id'])), len(set(val_df['subject_experiment_id'])) ,len(set(test_df['subject_experiment_id']))

In [ ]:
train_dataset = MultimodalFusionDataset(train_df, feature_store_path)
val_dataset = MultimodalFusionDataset(val_df, feature_store_path)
test_dataset = MultimodalFusionDataset(test_df, feature_store_path)

if USE_WEIGHTED_SAMPLER:
    sampler = torch.utils.data.WeightedRandomSampler(
        weights=torch.tensor(train_df["sample_weight"].values, dtype=torch.double),
        num_samples=len(train_df),
        replacement=True,
    )

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=NUM_WORKERS > 0, prefetch_factor=4)

else:
    # Shuffling matters for the fairness regularizer: batches must usually contain
    # multiple protected groups, otherwise the subgroup-MAE variance term is zero.
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=NUM_WORKERS > 0, prefetch_factor=4)

val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=NUM_WORKERS > 0, prefetch_factor=4)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=NUM_WORKERS > 0, prefetch_factor=4)


### Fairness-Aware Training

This copy trains matched baseline and fairness-aware fusion models on the same subject-level split. The regularizer is the thesis objective: the variance of subgroup MAEs within each mini-batch. Evaluation is always reported for both gender and age group, so a run regularized on one attribute can still reveal spillover or trade-offs on the other.


In [ ]:
# class MultimodalFusionTransformer(nn.Module):

#     def __init__(
#         self,
#         motion_input_dim,
#         hr_input_dim,
#         max_seq_len=SEQUENCE_LENGTH,
#         visual_feature_dim=VISUAL_FEATURE_DIM,
#         embed_dim=256,
#         num_heads=4,
#         num_layers=2,
#         dropout=0.2
#     ):
#         super().__init__()

#         self.max_seq_len = max_seq_len
#         self.num_modalities = 3

#         self.visual_projection = nn.Sequential(
#             nn.Linear(visual_feature_dim + 1, embed_dim),
#             nn.LayerNorm(embed_dim),
#             nn.ReLU(),
#             nn.Dropout(dropout)
#         )

#         self.motion_projection = nn.Sequential(
#             nn.Linear(motion_input_dim, 128),
#             nn.LayerNorm(128),
#             nn.ReLU(),
#             nn.Dropout(dropout),
#             nn.Linear(128, embed_dim),
#             nn.LayerNorm(embed_dim),
#             nn.ReLU()
#         )

#         self.hr_projection = nn.Sequential(
#             nn.Linear(hr_input_dim, 32),
#             nn.LayerNorm(32),
#             nn.ReLU(),
#             nn.Dropout(dropout),
#             nn.Linear(32, embed_dim),
#             nn.LayerNorm(embed_dim),
#             nn.ReLU()
#         )

#         self.time_embedding = nn.Parameter(torch.zeros(max_seq_len, embed_dim))
#         self.modality_embedding = nn.Parameter(torch.zeros(self.num_modalities, embed_dim))

#         encoder_layer = nn.TransformerEncoderLayer(
#             d_model=embed_dim,
#             nhead=num_heads,
#             dim_feedforward=512,
#             dropout=dropout,
#             batch_first=True
#         )

#         self.transformer = nn.TransformerEncoder(
#             encoder_layer,
#             num_layers=num_layers
#         )

#         # Learned modality pooling over final-step visual/motion/HR tokens.
#         self.modality_pool = nn.Sequential(
#             nn.Linear(embed_dim, 64),
#             nn.Tanh(),
#             nn.Linear(64, 1)
#         )

#         self.regressor = nn.Sequential(
#             nn.Linear(embed_dim, 128),
#             nn.ReLU(),
#             nn.Dropout(dropout),
#             nn.Linear(128, 1)
#         )

#     def forward(self, visual_features, motion, heart_rate, visual_missing_flags):
#         B, T, _ = visual_features.shape

#         visual_missing_flags = visual_missing_flags.unsqueeze(-1)
#         visual_features = torch.cat([visual_features, visual_missing_flags], dim=-1)

#         visual_tokens = self.visual_projection(visual_features)
#         motion_tokens = self.motion_projection(motion)
#         hr_tokens = self.hr_projection(heart_rate)

#         modality_tokens = torch.stack(
#             [visual_tokens, motion_tokens, hr_tokens],
#             dim=2
#         )  # B, T, 3, D

#         time_emb = self.time_embedding[:T].view(1, T, 1, -1)
#         modality_emb = self.modality_embedding.view(1, 1, self.num_modalities, -1)
#         modality_tokens = modality_tokens + time_emb + modality_emb

#         fused_tokens = modality_tokens.reshape(B, T * self.num_modalities, -1)

#         token_times = torch.arange(T, device=visual_features.device).repeat_interleave(self.num_modalities)
#         causal_mask = token_times.unsqueeze(0) > token_times.unsqueeze(1)

#         fused_tokens = self.transformer(
#             fused_tokens,
#             mask=causal_mask
#         )

#         final_step_tokens = fused_tokens[:, -self.num_modalities:, :]  # B, 3, D

#         pool_scores = self.modality_pool(final_step_tokens)            # B, 3, 1
#         pool_weights = torch.softmax(pool_scores, dim=1)               # B, 3, 1
#         final_token = (final_step_tokens * pool_weights).sum(dim=1)    # B, D

#         prediction = self.regressor(final_token)
#         return prediction.squeeze(1)

In [ ]:
class MultimodalLateFusionTransformer(nn.Module):

    def __init__(
        self,
        motion_input_dim,
        hr_input_dim,
        max_seq_len=SEQUENCE_LENGTH,
        visual_feature_dim=VISUAL_FEATURE_DIM,
        embed_dim=256,
        num_heads=4,
        num_layers=2,
        dropout=0.2
    ):
        super().__init__()

        self.max_seq_len = max_seq_len

        self.visual_projection = nn.Sequential(
            nn.Linear(visual_feature_dim + 1, embed_dim),
            nn.LayerNorm(embed_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        self.motion_projection = nn.Sequential(
            nn.Linear(motion_input_dim, 128),
            nn.LayerNorm(128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, embed_dim),
            nn.LayerNorm(embed_dim),
            nn.ReLU()
        )

        self.hr_projection = nn.Sequential(
            nn.Linear(hr_input_dim, 32),
            nn.LayerNorm(32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, embed_dim),
            nn.LayerNorm(embed_dim),
            nn.ReLU()
        )

        self.time_embedding = nn.Parameter(torch.zeros(max_seq_len, embed_dim))

        visual_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=1024,
            dropout=dropout,
            batch_first=True
        )

        sensor_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=512,
            dropout=dropout,
            batch_first=True
        )

        self.visual_encoder = nn.TransformerEncoder(
            visual_layer,
            num_layers=num_layers
        )

        self.motion_encoder = nn.TransformerEncoder(
            sensor_layer,
            num_layers=1
        )

        self.hr_encoder = nn.TransformerEncoder(
            sensor_layer,
            num_layers=1
        )

        self.fusion_gate = nn.Sequential(
            nn.Linear(embed_dim * 3, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 3)
        )

        self.regressor = nn.Sequential(
            nn.Linear(embed_dim * 3, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 1)
        )

    def forward(self, visual_features, motion, heart_rate, visual_missing_flags):
        B, T, _ = visual_features.shape

        visual_missing_flags = visual_missing_flags.unsqueeze(-1)
        visual_features = torch.cat([visual_features, visual_missing_flags], dim=-1)

        time_emb = self.time_embedding[:T].unsqueeze(0)

        visual_tokens = self.visual_projection(visual_features) + time_emb
        motion_tokens = self.motion_projection(motion) + time_emb
        hr_tokens = self.hr_projection(heart_rate) + time_emb

        token_times = torch.arange(T, device=visual_features.device)
        causal_mask = token_times.unsqueeze(0) > token_times.unsqueeze(1)

        visual_encoded = self.visual_encoder(visual_tokens, mask=causal_mask)
        motion_encoded = self.motion_encoder(motion_tokens, mask=causal_mask)
        hr_encoded = self.hr_encoder(hr_tokens, mask=causal_mask)

        visual_summary = visual_encoded[:, -1, :]
        motion_summary = motion_encoded[:, -1, :]
        hr_summary = hr_encoded[:, -1, :]

        summaries = torch.stack(
            [visual_summary, motion_summary, hr_summary],
            dim=1
        )  # B, 3, D

        fusion_input = torch.cat(
            [visual_summary, motion_summary, hr_summary],
            dim=1
        )  # B, 3D

        gate_logits = self.fusion_gate(fusion_input)          # B, 3
        gate_weights = torch.softmax(gate_logits, dim=1)      # B, 3

        gated_summaries = summaries * gate_weights.unsqueeze(-1)
        final_token = gated_summaries.reshape(B, -1)          # B, 3D

        prediction = self.regressor(final_token)
        return prediction.squeeze(1)

In [ ]:
def set_global_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_criterion(loss_type):
    if loss_type == "smooth_l1":
        return nn.SmoothL1Loss(beta=0.5, reduction="none"), "mae"
    if loss_type == "mse":
        return nn.MSELoss(reduction="none"), "rmse"
    if loss_type == "l1":
        return nn.L1Loss(reduction="none"), "mae"
    raise ValueError(f"Unknown LOSS_TYPE: {loss_type}")


def weighted_task_loss(per_sample_loss, sample_weights):
    if USE_WEIGHTED_LOSS:
        return (per_sample_loss * sample_weights).sum() / sample_weights.sum().clamp_min(1e-8)
    return per_sample_loss.mean()


def build_group_code_tensors(sequence_df, attributes):
    group_code_tensors = {}
    group_code_maps = {}

    for attribute in attributes:
        if attribute not in sequence_df.columns:
            continue

        values = sequence_df[attribute].astype("string").fillna("__missing__")
        groups = sorted([str(value) for value in values.unique() if str(value) != "__missing__"])
        mapping = {group: code for code, group in enumerate(groups)}
        codes = values.map(mapping).fillna(-1).astype(int).to_numpy()

        group_code_tensors[attribute] = torch.tensor(codes, dtype=torch.long)
        group_code_maps[attribute] = mapping

    return group_code_tensors, group_code_maps


def fairness_mae_variance_loss(
    preds,
    labels,
    idx,
    group_code_tensors,
    fairness_attributes,
    min_group_count=2,
):
    if not fairness_attributes or group_code_tensors is None:
        return preds.new_tensor(0.0)

    abs_errors = torch.abs(preds - labels)
    idx_cpu = idx.detach().cpu().long()
    attribute_losses = []

    for attribute in fairness_attributes:
        if attribute not in group_code_tensors:
            continue

        group_codes = group_code_tensors[attribute][idx_cpu].to(preds.device)
        group_maes = []

        for group_code in torch.unique(group_codes):
            if int(group_code.item()) < 0:
                continue
            mask = group_codes == group_code
            if int(mask.sum().item()) >= min_group_count:
                group_maes.append(abs_errors[mask].mean())

        if len(group_maes) >= 2:
            group_maes = torch.stack(group_maes)
            attribute_losses.append(torch.mean((group_maes - group_maes.mean()) ** 2))

    if not attribute_losses:
        return preds.new_tensor(0.0)

    return torch.stack(attribute_losses).mean()


def train_one_epoch(
    model,
    loader,
    optimizer,
    criterion,
    fairness_attributes=None,
    fairness_lambda=0.0,
    group_code_tensors=None,
    fairness_min_group_count=2,
):
    model.train()
    total_loss = 0.0
    total_task_loss = 0.0
    total_fairness_loss = 0.0
    preds_all = []
    labels_all = []
    pbar = tqdm(loader, desc="Training", leave=False)

    for visual_features, motion, heart_rate, visual_missing_flags, labels, sample_weights, idx in pbar:
        visual_features = visual_features.to(device)
        motion = motion.to(device)
        heart_rate = heart_rate.to(device)
        visual_missing_flags = visual_missing_flags.to(device)
        labels = labels.to(device)
        sample_weights = sample_weights.to(device)

        preds = model(visual_features, motion, heart_rate, visual_missing_flags)
        per_sample_loss = criterion(preds, labels)
        task_loss = weighted_task_loss(per_sample_loss, sample_weights)
        fairness_loss = fairness_mae_variance_loss(
            preds=preds,
            labels=labels,
            idx=idx,
            group_code_tensors=group_code_tensors,
            fairness_attributes=fairness_attributes,
            min_group_count=fairness_min_group_count,
        )
        loss = task_loss + float(fairness_lambda) * fairness_loss

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += float(loss.item())
        total_task_loss += float(task_loss.item())
        total_fairness_loss += float(fairness_loss.item())
        preds_all.extend(preds.detach().cpu().numpy())
        labels_all.extend(labels.detach().cpu().numpy())

    train_mae = mean_absolute_error(labels_all, preds_all)
    train_rmse = np.sqrt(mean_squared_error(labels_all, preds_all))

    return {
        "loss": total_loss / len(loader),
        "task_loss": total_task_loss / len(loader),
        "fairness_loss": total_fairness_loss / len(loader),
        "mae": train_mae,
        "rmse": train_rmse,
    }


def evaluate(model, loader):
    model.eval()
    preds_all = []
    labels_all = []
    inference_times = []
    pbar = tqdm(loader, desc="Evaluating", leave=False)

    with torch.no_grad():
        for visual_features, motion, heart_rate, visual_missing_flags, labels, sample_weights, idx in pbar:
            visual_features = visual_features.to(device)
            motion = motion.to(device)
            heart_rate = heart_rate.to(device)
            visual_missing_flags = visual_missing_flags.to(device)
            labels = labels.to(device)

            start = time.time()
            preds = model(visual_features, motion, heart_rate, visual_missing_flags)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            end = time.time()

            inference_times.append(end - start)
            preds_all.extend(preds.detach().cpu().numpy())
            labels_all.extend(labels.detach().cpu().numpy())

    return {
        "mae": mean_absolute_error(labels_all, preds_all),
        "rmse": np.sqrt(mean_squared_error(labels_all, preds_all)),
        "avg_time": np.mean(inference_times),
    }


In [ ]:
def evaluate_legacy_tuple(model, loader):

    model.eval()
    preds_all = []
    labels_all = []
    inference_times = []
    pbar = tqdm(loader, desc="Evaluating", leave=False)

    with torch.no_grad():
        for visual_features, motion, heart_rate, visual_missing_flags, labels, sample_weights, idx in pbar:
            visual_features = visual_features.to(device)
            motion = motion.to(device)
            heart_rate = heart_rate.to(device)
            visual_missing_flags = visual_missing_flags.to(device)
            labels = labels.to(device)

            start = time.time()
            preds = model(visual_features, motion, heart_rate, visual_missing_flags)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            end = time.time()

            inference_times.append(end - start)
            preds_all.extend(preds.cpu().numpy())
            labels_all.extend(labels.cpu().numpy())

    mae = mean_absolute_error(labels_all, preds_all)
    rmse = np.sqrt(mean_squared_error(labels_all, preds_all))

    return mae, rmse, np.mean(inference_times)

In [ ]:
import copy

class EarlyStopping:
    def __init__(self, patience=5, min_delta=0, model_path=None):
        self.patience = patience
        self.min_delta = min_delta
        self.best_score = float("inf")
        self.counter = 0
        self.best_model = None
        self.best_epoch = None
        self.model_path = model_path

    def step(self, metric, model, epoch):
        if metric < self.best_score - self.min_delta:
            self.best_score = metric
            self.counter = 0

            self.best_model = copy.deepcopy(model.state_dict())
            self.best_epoch = epoch

            if self.model_path is not None:
                torch.save(self.best_model, self.model_path)

            return False  # continue

        else:
            self.counter += 1
            print(f"EarlyStopping counter: {self.counter}/{self.patience}")

            return self.counter >= self.patience

In [ ]:
LOSS_TYPE = "smooth_l1"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

learning_rate = 1e-4
weight_decay = 1e-5
NUM_EPOCHS = 50
PATIENCE = 7

# Keep this at [42] for a quick smoke run. For thesis stability evidence, use
# several seeds and interpret the seed_stability_table produced below.
# RUN_SEEDS = [42]
RUN_SEEDS = [42, 2025, 2026, 2027]

# Lambda sweep for the fairness regularizer. Baseline is trained once per seed
# with lambda=0.0; each fairness configuration is trained for every value here.
FAIRNESS_LAMBDAS = [0.05, 0.1, 0.2, 0.5]

FAIRNESS_MIN_GROUP_COUNT = 2
FAIRNESS_EVAL_ATTRIBUTES = ["gender", "age_group"]

BASELINE_RUN_CONFIG = {
    "config_id": "baseline",
    "label": "Baseline",
    "fairness_attributes": [],
    "fairness_lambda": 0.0,
}

FAIRNESS_ATTRIBUTE_CONFIGS = [
    {
        "config_id": "fair_gender",
        "label": "Fairness regularizer: gender",
        "fairness_attributes": ["gender"],
    },
    {
        "config_id": "fair_age",
        "label": "Fairness regularizer: age group",
        "fairness_attributes": ["age_group"],
    },
    {
        "config_id": "fair_gender_age",
        "label": "Fairness regularizer: gender + age group",
        "fairness_attributes": ["gender", "age_group"],
    },
]

FAIRNESS_RUN_CONFIGS = [BASELINE_RUN_CONFIG] + [
    {**config, "fairness_lambda": fairness_lambda}
    for fairness_lambda in FAIRNESS_LAMBDAS
    for config in FAIRNESS_ATTRIBUTE_CONFIGS
]

criterion, EARLY_STOPPING_METRIC = make_criterion(LOSS_TYPE)


def make_model():
    return MultimodalLateFusionTransformer(
        motion_input_dim=MOTION_INPUT_DIM,
        hr_input_dim=HR_INPUT_DIM,
        max_seq_len=SEQUENCE_LENGTH,
        visual_feature_dim=VISUAL_FEATURE_DIM,
        embed_dim=256,
        num_heads=4,
        num_layers=2,
        dropout=0.2,
    )


def make_optimizer_and_scheduler(model):
    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=learning_rate,
        weight_decay=weight_decay,
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.3,
        patience=3,
    )
    return optimizer, scheduler


print(f"Loss type: {LOSS_TYPE}")
print(f"Early stopping / checkpoint metric: validation {EARLY_STOPPING_METRIC.upper()}")
print("Fairness configs:")
for config in FAIRNESS_RUN_CONFIGS:
    print(f"  {config['config_id']} | lambda={config['fairness_lambda']} | attrs={config['fairness_attributes']}")


In [ ]:
base_name = f"056.fusion_split_hr_transformer_{LOSS_TYPE}_{FEATURES}_seq{SEQUENCE_LENGTH}"
if USE_WEIGHTED_LOSS:
    base_name += f"_weighted_loss_a{WEIGHT_ALPHA}"
if USE_WEIGHTED_SAMPLER:
    base_name += f"_weighted_sampler_a{WEIGHT_ALPHA}"

fairness_results_dir = "results/Multimodal Fusion Fairness"
fairness_model_dir = "models/Multimodal Fusion Fairness"
fairness_fig_dir = "figs/Multimodal Fusion Fairness"
os.makedirs(fairness_results_dir, exist_ok=True)
os.makedirs(fairness_model_dir, exist_ok=True)
os.makedirs(fairness_fig_dir, exist_ok=True)


def make_run_name(config, run_seed):
    attrs = "none" if not config["fairness_attributes"] else "_".join(config["fairness_attributes"])
    lam = str(config["fairness_lambda"]).replace(".", "p")
    return f"{base_name}_{config['config_id']}_{attrs}_lambda{lam}_seed{run_seed}"


def make_run_paths(run_name):
    return {
        "model_path": os.path.join(fairness_model_dir, f"{run_name}.pt"),
        "results_path": os.path.join(fairness_results_dir, f"{run_name}_results.json"),
        "prediction_path": os.path.join(fairness_results_dir, f"{run_name}_test_predictions.csv"),
        "fig_path": os.path.join(fairness_fig_dir, f"{run_name}_training_curves.png"),
    }


all_fairness_attributes = sorted(
    set(FAIRNESS_EVAL_ATTRIBUTES).union(
        attribute
        for config in FAIRNESS_RUN_CONFIGS
        for attribute in config["fairness_attributes"]
    )
)
group_code_tensors, group_code_maps = build_group_code_tensors(train_dataset.df, all_fairness_attributes)
group_code_maps


In [ ]:
fairness_training_runs = []

for run_seed in RUN_SEEDS:
    for config in FAIRNESS_RUN_CONFIGS:
        # Reset before every configuration so paired baseline/fairness runs for
        # the same seed differ by objective, not by initialization or shuffle order.
        set_global_seed(run_seed)

        run_name = make_run_name(config, run_seed)
        run_paths = make_run_paths(run_name)

        model = make_model().to(device)
        optimizer, scheduler = make_optimizer_and_scheduler(model)
        early_stopping = EarlyStopping(patience=PATIENCE, model_path=run_paths["model_path"])

        history = {
            "epoch": [],
            "train_loss": [],
            "train_task_loss": [],
            "train_fairness_loss": [],
            "train_mae": [],
            "train_rmse": [],
            "val_mae": [],
            "val_rmse": [],
            "monitor_metric": [],
        }

        training_start = time.time()
        print(f"\n=== {run_name} ===")

        for epoch in range(NUM_EPOCHS):
            train_metrics = train_one_epoch(
                model=model,
                loader=train_loader,
                optimizer=optimizer,
                criterion=criterion,
                fairness_attributes=config["fairness_attributes"],
                fairness_lambda=config["fairness_lambda"],
                group_code_tensors=group_code_tensors,
                fairness_min_group_count=FAIRNESS_MIN_GROUP_COUNT,
            )
            val_metrics = evaluate(model, val_loader)

            monitor_value = val_metrics[EARLY_STOPPING_METRIC]

            history["epoch"].append(epoch + 1)
            history["train_loss"].append(train_metrics["loss"])
            history["train_task_loss"].append(train_metrics["task_loss"])
            history["train_fairness_loss"].append(train_metrics["fairness_loss"])
            history["train_mae"].append(train_metrics["mae"])
            history["train_rmse"].append(train_metrics["rmse"])
            history["val_mae"].append(val_metrics["mae"])
            history["val_rmse"].append(val_metrics["rmse"])
            history["monitor_metric"].append(monitor_value)

            print(f"Epoch {epoch + 1:02d} | train MAE {train_metrics['mae']:.4f} | val MAE {val_metrics['mae']:.4f} | fair loss {train_metrics['fairness_loss']:.6f}")

            scheduler.step(monitor_value)
            stop = early_stopping.step(monitor_value, model, epoch)

            if stop:
                print("Early stopping triggered")
                break

        training_end = time.time()
        best_epoch_index = early_stopping.best_epoch
        if best_epoch_index is None:
            best_epoch_index = int(np.argmin(history["monitor_metric"]))

        fairness_training_runs.append({
            "run_name": run_name,
            "run_seed": run_seed,
            "config_id": config["config_id"],
            "label": config["label"],
            "fairness_attributes": config["fairness_attributes"],
            "fairness_lambda": config["fairness_lambda"],
            "best_epoch": int(best_epoch_index) + 1,
            "num_epochs_run": len(history["epoch"]),
            "training_time_sec": round(float(training_end - training_start), 2),
            "history": history,
            **run_paths,
        })

        # Keep the most recent run available for the optional curve cell below.
        train_mae_list = history["train_mae"]
        train_rmse_list = history["train_rmse"]
        val_mae_list = history["val_mae"]
        val_rmse_list = history["val_rmse"]
        best_epoch = best_epoch_index
        model_path = run_paths["model_path"]
        model_figs = run_paths["fig_path"]

pd.DataFrame([
    {
        key: value
        for key, value in run.items()
        if key not in {"history"}
    }
    for run in fairness_training_runs
])


### Evaluation

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

for run in fairness_training_runs:
    history = run["history"]
    label = f"{run['config_id']} seed={run['run_seed']}"
    ax[0].plot(history["epoch"], history["val_rmse"], label=label)
    ax[1].plot(history["epoch"], history["val_mae"], label=label)

ax[0].set_title("Validation RMSE")
ax[0].set_xlabel("Epoch")
ax[0].set_ylabel("RMSE")
ax[0].grid()

ax[1].set_title("Validation MAE")
ax[1].set_xlabel("Epoch")
ax[1].set_ylabel("MAE")
ax[1].grid()

for axis in ax:
    axis.legend(fontsize=8)

plt.tight_layout()
plt.show()


In [ ]:
from importlib import reload
import src.evaluation as ev

ev = reload(ev)

In [ ]:
fairness_predictions_by_model = {}
fairness_results = []

for run in fairness_training_runs:
    model = make_model().to(device)
    model.load_state_dict(torch.load(run["model_path"], map_location=device))
    model.eval()

    results_df, best_mae, best_rmse, best_latency_batch, best_latency_sample = ev.collect_predictions(
        model=model,
        df=test_df,
        loader=test_loader,
        device=device,
        mode="fusion",
    )

    results_df = ev.add_robustness_metadata(
        results_df,
        sequence_df=test_df,
        visual_missing_col="visual_missing_flags",
    )

    ev.save_prediction_frame(results_df, run["prediction_path"])
    fairness_predictions_by_model[run["run_name"]] = results_df

    result = ev.evaluate_predictions(results_df, model_name=run["run_name"])
    result.update({
        "run_id": run["run_name"],
        "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
        "experiment_name": run["run_name"],
        "model_path": run["model_path"],
        "results_path": run["results_path"],
        "prediction_path": run["prediction_path"],
        "loss_type": LOSS_TYPE,
        "features": FEATURES,
        "sequence_length": SEQUENCE_LENGTH,
        "stratify": STRATIFY_COLUMN,
        "run_seed": run["run_seed"],
        "config_id": run["config_id"],
        "fairness_attributes": run["fairness_attributes"],
        "fairness_lambda": run["fairness_lambda"],
        "fairness_min_group_count": FAIRNESS_MIN_GROUP_COUNT,
        "best_epoch": run["best_epoch"],
        "num_epochs_run": run["num_epochs_run"],
        "training_time_sec": run["training_time_sec"],
        "learning_rate": learning_rate,
        "weight_decay": weight_decay,
        "latency_per_batch": round(float(best_latency_batch), 4),
        "latency_per_sample_ms": round(float(best_latency_sample * 1000), 4),
    })

    with open(run["results_path"], "w") as f:
        json.dump(result, f, indent=4, default=str)

    fairness_results.append(result)

fairness_overall_table = ev.overall_table(fairness_predictions_by_model)
fairness_table = ev.fairness_table(fairness_predictions_by_model)

display(fairness_overall_table.round(3))
display(fairness_table.round(3))


In [ ]:
PAIR_KEYS = ["subject_experiment_id", "time_sec"]
BOOTSTRAP_RUNS = 100
BOOTSTRAP_CLUSTER_COL = "subject_experiment_id"
BOOTSTRAP_SEED = SEED


def _paired_frame(baseline_df, candidate_df):
    base_cols = list(dict.fromkeys(PAIR_KEYS + ["true", "pred", "gender", "age_group"]))
    base = baseline_df[base_cols].copy()
    cand = candidate_df[PAIR_KEYS + ["pred"]].copy()
    return base.merge(cand, on=PAIR_KEYS, how="inner", suffixes=("_baseline", "_candidate"))


def _metrics_from_pair(pair_df, pred_col, attribute):
    abs_error = np.abs(pair_df[pred_col].astype(float) - pair_df["true"].astype(float))
    overall_mae = float(abs_error.mean())
    group_mae = (
        pair_df.assign(abs_error=abs_error)
        .dropna(subset=[attribute])
        .groupby(attribute, observed=True)["abs_error"]
        .mean()
        .dropna()
    )
    if group_mae.empty:
        return {"mae": overall_mae, "worst_group_mae": np.nan, "gap": np.nan}
    return {
        "mae": overall_mae,
        "worst_group_mae": float(group_mae.max()),
        "gap": float(group_mae.max() - group_mae.min()),
    }


def _paired_delta_row(pair_df, attribute):
    baseline = _metrics_from_pair(pair_df, "pred_baseline", attribute)
    candidate = _metrics_from_pair(pair_df, "pred_candidate", attribute)

    return {
        "attribute": attribute,
        "baseline_mae": baseline["mae"],
        "candidate_mae": candidate["mae"],
        "overall_mae_gain": baseline["mae"] - candidate["mae"],
        "baseline_worst_group_mae": baseline["worst_group_mae"],
        "candidate_worst_group_mae": candidate["worst_group_mae"],
        "worst_group_mae_gain": baseline["worst_group_mae"] - candidate["worst_group_mae"],
        "baseline_gap": baseline["gap"],
        "candidate_gap": candidate["gap"],
        "gap_reduction": baseline["gap"] - candidate["gap"],
    }


def paired_bootstrap_delta(pair_df, attribute, n_boot=100, seed=42):
    rng = np.random.default_rng(seed)
    clusters = pair_df[BOOTSTRAP_CLUSTER_COL].dropna().unique()
    grouped = {cluster: group for cluster, group in pair_df.groupby(BOOTSTRAP_CLUSTER_COL, sort=False)}
    rows = []

    for _ in range(n_boot):
        sampled_clusters = rng.choice(clusters, size=len(clusters), replace=True)
        sampled = pd.concat([grouped[cluster] for cluster in sampled_clusters], ignore_index=True)
        rows.append(_paired_delta_row(sampled, attribute))

    boot_df = pd.DataFrame(rows)
    metrics = ["overall_mae_gain", "worst_group_mae_gain", "gap_reduction"]
    summary = pd.DataFrame({
        "metric": metrics,
        "mean": [boot_df[metric].mean() for metric in metrics],
        "ci_low": [boot_df[metric].quantile(0.025) for metric in metrics],
        "ci_high": [boot_df[metric].quantile(0.975) for metric in metrics],
    })
    return summary, boot_df


run_lookup = {(run["run_seed"], run["config_id"]): run for run in fairness_training_runs}
pairwise_rows = []
bootstrap_rows = []

for run in fairness_training_runs:
    if run["config_id"] == "baseline":
        continue

    baseline_run = run_lookup.get((run["run_seed"], "baseline"))
    if baseline_run is None:
        continue

    pair_df = _paired_frame(
        fairness_predictions_by_model[baseline_run["run_name"]],
        fairness_predictions_by_model[run["run_name"]],
    )

    for attribute in FAIRNESS_EVAL_ATTRIBUTES:
        row = _paired_delta_row(pair_df, attribute)
        row.update({
            "run_seed": run["run_seed"],
            "baseline_model": baseline_run["run_name"],
            "candidate_model": run["run_name"],
            "candidate_config": run["config_id"],
            "candidate_fairness_attributes": ",".join(run["fairness_attributes"]),
            "fairness_lambda": run["fairness_lambda"],
            "n_paired_samples": len(pair_df),
        })
        pairwise_rows.append(row)

        summary, boot_df = paired_bootstrap_delta(
            pair_df,
            attribute=attribute,
            n_boot=BOOTSTRAP_RUNS,
            seed=BOOTSTRAP_SEED + int(run["run_seed"]) + len(pairwise_rows),
        )
        summary.insert(0, "attribute", attribute)
        summary.insert(0, "fairness_lambda", run["fairness_lambda"])
        summary.insert(0, "candidate_config", run["config_id"])
        summary.insert(0, "run_seed", run["run_seed"])
        bootstrap_rows.append(summary)

fairness_pairwise_table = pd.DataFrame(pairwise_rows)
bootstrap_delta_summary = pd.concat(bootstrap_rows, ignore_index=True) if bootstrap_rows else pd.DataFrame()

seed_stability_table = (
    fairness_pairwise_table
    .groupby(["candidate_config", "fairness_lambda", "attribute"], observed=True)
    .agg(
        seeds=("run_seed", "nunique"),
        mean_overall_mae_gain=("overall_mae_gain", "mean"),
        sd_overall_mae_gain=("overall_mae_gain", "std"),
        mean_worst_group_mae_gain=("worst_group_mae_gain", "mean"),
        sd_worst_group_mae_gain=("worst_group_mae_gain", "std"),
        worst_group_gain_positive_rate=("worst_group_mae_gain", lambda s: float((s > 0).mean())),
        mean_gap_reduction=("gap_reduction", "mean"),
        sd_gap_reduction=("gap_reduction", "std"),
        gap_reduction_positive_rate=("gap_reduction", lambda s: float((s > 0).mean())),
    )
    .reset_index()
)

display(fairness_pairwise_table.round(4))
display(bootstrap_delta_summary.round(4))
display(seed_stability_table.round(4))


In [ ]:
import seaborn as sns

plot_table = fairness_pairwise_table.copy()
plot_table["worst_group_mae_gain"] = plot_table["worst_group_mae_gain"].astype(float)
plot_table["gap_reduction"] = plot_table["gap_reduction"].astype(float)
plot_table["config_lambda"] = plot_table.apply(
    lambda row: f"{row['candidate_config']} | lambda={row['fairness_lambda']}",
    axis=1,
)

fig, ax = plt.subplots(1, 2, figsize=(14, 8), sharey=True)

sns.barplot(
    data=plot_table,
    y="config_lambda",
    x="worst_group_mae_gain",
    hue="attribute",
    ax=ax[0],
)
ax[0].axvline(0, color="black", linewidth=1)
ax[0].set_title("Worst-group MAE gain vs baseline")
ax[0].set_xlabel("Positive means lower worst-group MAE")
ax[0].set_ylabel("Fairness configuration")

sns.barplot(
    data=plot_table,
    y="config_lambda",
    x="gap_reduction",
    hue="attribute",
    ax=ax[1],
)
ax[1].axvline(0, color="black", linewidth=1)
ax[1].set_title("Subgroup gap reduction vs baseline")
ax[1].set_xlabel("Positive means smaller subgroup gap")
ax[1].set_ylabel("")

plt.tight_layout()
plt.show()


In [ ]:
fairness_pairwise_path = os.path.join(fairness_results_dir, "fairness_pairwise_comparison.csv")
bootstrap_delta_path = os.path.join(fairness_results_dir, "fairness_paired_bootstrap_deltas.csv")
seed_stability_path = os.path.join(fairness_results_dir, "fairness_seed_stability.csv")

fairness_pairwise_table.to_csv(fairness_pairwise_path, index=False)
bootstrap_delta_summary.to_csv(bootstrap_delta_path, index=False)
seed_stability_table.to_csv(seed_stability_path, index=False)

print(f"Saved paired comparison: {fairness_pairwise_path}")
print(f"Saved paired bootstrap deltas: {bootstrap_delta_path}")
print(f"Saved seed stability: {seed_stability_path}")


In [ ]:
def interpret_fairness_row(row):
    if row["worst_group_mae_gain"] > 0 and row["gap_reduction"] > 0:
        return "genuine worst-group improvement"
    if row["worst_group_mae_gain"] <= 0 and row["gap_reduction"] > 0:
        return "gap reduction without worst-group gain"
    if row["worst_group_mae_gain"] > 0 and row["gap_reduction"] <= 0:
        return "worst-group improved but disparity did not shrink"
    return "no fairness improvement"


answer_table = fairness_pairwise_table.copy()
answer_table["interpretation"] = answer_table.apply(interpret_fairness_row, axis=1)
answer_table = answer_table[
    [
        "candidate_config",
        "fairness_lambda",
        "run_seed",
        "attribute",
        "overall_mae_gain",
        "worst_group_mae_gain",
        "gap_reduction",
        "interpretation",
    ]
].sort_values(["candidate_config", "fairness_lambda", "attribute", "run_seed"])

display(answer_table.round(4))


In [ ]:
answer_table


In [ ]:
# Notes for the thesis write-up
#
# 1. Use a fixed subject-level split when comparing baseline and fairness-aware
#    models. Changing STRATIFY_COLUMN changes the held-out subjects, so it should
#    be treated as a split-sensitivity experiment rather than proof that the
#    regularizer helped or failed.
# 2. A fairness-aware run is strong only when worst_group_mae_gain is positive
#    and gap_reduction is positive for the protected attribute of interest.
#    A positive gap_reduction with zero or negative worst_group_mae_gain means
#    the regularizer mostly compressed group differences without helping the
#    hardest group.
# 3. For the final robustness claim, set RUN_SEEDS to several values and inspect
#    seed_stability_table plus bootstrap_delta_summary. Stable effects should
#    have positive-rate near 1.0 and bootstrap intervals mostly above zero.
